# Heart Disease Prediction

Author: Lucy Giang (Jan. 2026)

This project seeks to build and evaluate baseline machine learning models using the [UCI Heart Disease dataset](https://www.kaggle.com/datasets/redwankarimsony/heart-disease-data). The prediction target is **presence of heart disease** (`num > 0`) versus **absence** (`num == 0`). This notebook is for learning and analysis, not clinical decision-making.

## Setup

Install the packages with `pip install pandas numpy matplotlib seaborn scikit-learn`. The CSV is expected at `data/heart_disease_uci.csv` relative to this notebook.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, classification_report,
    ConfusionMatrixDisplay, confusion_matrix, f1_score,
    precision_score, recall_score, roc_auc_score, RocCurveDisplay,
)
from sklearn.model_selection import train_test_split, cross_validate, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_STATE = 42
DATA_PATH = Path('data/heart_disease_uci.csv')
sns.set_theme(style='whitegrid')

## Load The Dataset

Note: Download the dataset first from Kaggle.

In [ ]:
df = pd.read_csv(DATA_PATH)
print(f'Shape: {df.shape[0]} rows × {df.shape[1]} columns')
df.head()

In [ ]:
df.info()

In [ ]:
display(df.describe(include='all').T)

missing = df.isna().sum().sort_values(ascending=False)
missing[missing > 0].rename('missing values').to_frame()

The file includes measurements from multiple source sites. It also includes an `id` field and a `dataset` site label. We exclude both from predictors: `id` is an identifier, while site can act as a shortcut and may not generalize to a new source. Missing values are imputed within each model pipeline.

## Define the prediction target

`num` is the original diagnosis severity label (0 through 4). For this binary task, any value above 0 counts as disease present. We keep the original `num` column out of the features.

In [ ]:
TARGET = 'has_disease'
df[TARGET] = (df['num'] > 0).astype('int8')

print('Original num values:')
print(df['num'].value_counts().sort_index())
print('\nBinary target counts (0 = absent, 1 = present):')
print(df[TARGET].value_counts().sort_index())
print(f'\nPositive class prevalence: {df[TARGET].mean():.1%}')

In [ ]:
ax = df[TARGET].value_counts().sort_index().plot.bar(color=['steelblue', 'salmon'])
ax.set_xticks([0, 1], labels=['Absent (0)', 'Present (1)'], rotation=0)
ax.set_ylabel('Number of records')
ax.set_title('Target class balance')
plt.show()

## Train/test split

The test set is held back until model selection is complete. Stratification preserves the class ratio in each split.

In [ ]:
X = df.drop(columns=['num', TARGET, 'id', 'dataset'])
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

print(f'Train: {X_train.shape}; Test: {X_test.shape}')
print(f'Train positive prevalence: {y_train.mean():.1%}')
print(f'Test positive prevalence: {y_test.mean():.1%}')
print('Features:', list(X.columns))

## Preprocessing and model pipelines

Numeric columns receive median imputation and scaling. Categorical columns receive most-frequent imputation and one-hot encoding. Because these steps are fitted inside the pipeline, validation folds and the test set do not influence training-time preprocessing.

In [ ]:
numeric_features = X_train.select_dtypes(include=['number']).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=['number']).columns.tolist()

numeric_preprocessor = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
])
categorical_preprocessor = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore')),
])
preprocessor = ColumnTransformer(transformers=[
    ('numeric', numeric_preprocessor, numeric_features),
    ('categorical', categorical_preprocessor, categorical_features),
])

models = {
    'Logistic Regression': Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', LogisticRegression(max_iter=2000, class_weight='balanced', random_state=RANDOM_STATE)),
    ]),
    'Random Forest': Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', RandomForestClassifier(n_estimators=400, min_samples_leaf=2,
                                              class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1)),
    ]),
}
print('Numeric:', numeric_features)
print('Categorical:', categorical_features)

## Compare models with cross-validation on the training set

We compare ROC AUC, which measures ranking across thresholds, and F1, which balances precision and recall at the default decision threshold. In a health context these metrics do not express clinical utility; false negatives and false positives may have different costs.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = {'roc_auc': 'roc_auc', 'f1': 'f1', 'balanced_accuracy': 'balanced_accuracy'}

cv_rows = []
for name, model in models.items():
    scores = cross_validate(model, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1)
    cv_rows.append({
        'Model': name,
        **{f'{metric} mean': scores[f'test_{metric}'].mean() for metric in scoring},
        **{f'{metric} std': scores[f'test_{metric}'].std() for metric in scoring},
    })
cv_results = pd.DataFrame(cv_rows).sort_values('roc_auc mean', ascending=False)
display(cv_results)

## Tune the selected model with cross-validation
Choose the model family using the baseline cross-validation results, then search a compact parameter grid for that family. Grid search uses only the training split; the test set remains untouched until the final evaluation. ROC AUC is the selection metric.


In [ ]:
from sklearn.model_selection import GridSearchCV

best_name = cv_results.iloc[0]['Model']
base_pipeline = models[best_name]

if best_name == 'Logistic Regression':
    param_grid = {
        'classifier__C': [0.01, 0.1, 1.0, 10.0],
        'classifier__solver': ['lbfgs'],
    }
else:
    param_grid = {
        'classifier__n_estimators': [200, 400],
        'classifier__min_samples_leaf': [1, 2, 4],
        'classifier__max_features': ['sqrt', 0.8],
    }

grid_search = GridSearchCV(
    estimator=base_pipeline,
    param_grid=param_grid,
    scoring='roc_auc',
    cv=cv,
    n_jobs=-1,
    refit=True,
)
grid_search.fit(X_train, y_train)
best_model = grid_search.best_estimator_  # fitted preprocessing + classifier pipeline

print(f'Model family: {best_name}')
print('Best parameters:', grid_search.best_params_)
print(f'Best mean CV ROC AUC: {grid_search.best_score_:.3f}')

y_pred = best_model.predict(X_test)
y_score = best_model.predict_proba(X_test)[:, 1]

print('\nHeld-out test metrics:')
print(f'Accuracy:          {accuracy_score(y_test, y_pred):.3f}')
print(f'Balanced accuracy: {balanced_accuracy_score(y_test, y_pred):.3f}')
print(f'Precision:         {precision_score(y_test, y_pred):.3f}')
print(f'Recall:            {recall_score(y_test, y_pred):.3f}')
print(f'F1:                {f1_score(y_test, y_pred):.3f}')
print(f'ROC AUC:           {roc_auc_score(y_test, y_score):.3f}')
print('\nClassification report:')
print(classification_report(y_test, y_pred, target_names=['Absent', 'Present']))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, display_labels=['Absent', 'Present'],
                                        cmap='Blues', colorbar=False, ax=axes[0])
axes[0].set_title('Confusion matrix')
RocCurveDisplay.from_predictions(y_test, y_score, ax=axes[1], name=best_name)
axes[1].plot([0, 1], [0, 1], linestyle='--', color='gray', label='Chance')
axes[1].set_title('ROC curve')
axes[1].legend()
plt.tight_layout()
plt.show()

## Interpret the fitted model
For logistic regression, coefficients are shown on the preprocessed feature scale (numeric variables are standardized and categorical variables are one-hot encoded). For random forest, impurity-based importance is shown; it can favor high-cardinality features and should be treated as descriptive rather than causal.


In [ ]:
fitted_preprocessor = best_model.named_steps['preprocessor']
fitted_classifier = best_model.named_steps['classifier']
feature_names = fitted_preprocessor.get_feature_names_out()

if best_name == 'Logistic Regression':
    importance = pd.Series(fitted_classifier.coef_[0], index=feature_names)
    importance = importance.reindex(importance.abs().sort_values(ascending=False).head(15).index)
    plot_title = 'Largest logistic regression coefficients'
else:
    importance = pd.Series(fitted_classifier.feature_importances_, index=feature_names)
    importance = importance.sort_values(ascending=False).head(15).sort_values()
    plot_title = 'Top random forest feature importances'

colors = ['steelblue' if value < 0 else 'salmon' for value in importance.values]
ax = importance.sort_values().plot.barh(figsize=(9, 6), color=colors)
ax.set_title(plot_title)
ax.set_xlabel('Coefficient' if best_name == 'Logistic Regression' else 'Impurity-based importance')
ax.set_ylabel('')
plt.tight_layout()
plt.show()


## Save the final pipeline
Save the fitted pipeline, which includes preprocessing and the tuned classifier. The output path is relative to the notebook working directory. 


In [ ]:
import joblib

MODEL_PATH = Path('heart_disease_model.joblib')
joblib.dump(best_model, MODEL_PATH)
print(f'Saved fitted pipeline to {MODEL_PATH.resolve()}')


## Note:
- The default 0.5 classification threshold is only a starting point. Select a threshold based on the intended use and the relative costs of false negatives and false positives.
- Results depend on the random split and how the source file was assembled. Consider external validation or a site-aware evaluation before making general claims.
- This educational model is not a medical device and should not be used to diagnose or guide treatment.
